# Openjutsu (G4 / MiniMax H3 Motion Transfer, Video Edit, Face & Colour Fix, FLUX.3 Picture Prep)

Co-locates ComfyUI (`127.0.0.1:8188` + 13 custom node packs), patched `comfy-api-proxy` (`127.0.0.1:8189`), and Openjutsu's Next.js 16 web UI (`127.0.0.1:3000`) on a Colab G4 (RTX PRO 6000 Blackwell, 96 GB VRAM) behind Caddy Basic Auth + session cookie + Bearer token. Models live on fast local NVMe SSD (`/content/ComfyUI/models`); generated videos, clip checkpoints, inputs, and custom LoRAs persist on Google Drive.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "openjutsu")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "3000"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- ComfyUI + 13 custom node packs + comfy-api-proxy + Openjutsu Next.js + local SSD models
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import hashlib, platform

OJ_WORK = Path(os.environ.get('CT_OPENJUTSU_WORK_DIR', '/content/openjutsu'))
OJ_COMFY = Path(os.environ.get('CT_COMFY_DIR', '/content/ComfyUI'))
OJ_PROFILE = (os.environ.get('CT_OPENJUTSU_PROFILE', 'full').strip().lower() or 'full')
OJ_WORK.mkdir(parents=True, exist_ok=True)
(OJ_WORK / 'colab_cache.py').write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding='utf-8')
(OJ_WORK / 'openjutsu_runtime.py').write_text('"""Self-contained Colab setup and verification helpers for Openjutsu (ComfyUI + comfy-api-proxy + Next.js)."""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport hashlib\nimport importlib\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\nNODE_VERSION = "22.14.0"\nPROXY_VERSION = "0.1.6"\nDEFAULT_OPENJUTSU_REPO = "https://github.com/deluxebear/colab_openjutsu.git"\nHF_RESOLVE_URL_RE = re.compile(r"^https?://huggingface\\.co/([^/]+/[^/]+)/(?:resolve|blob)/([^/]+)/(.+?)(?:\\?.*)?$")\n\nCUSTOM_NODES: tuple[tuple[str, str], ...] = (\n    ("ComfyUI-MiniMaxH3-Context-Loop", "https://github.com/ethanfel/ComfyUI-MiniMaxH3-Context-Loop.git"),\n    ("comfyui-h3-oneclick-swap", "https://github.com/D3voz/comfyui-h3-oneclick-swap.git"),\n    ("ComfyUI-Fantastic-MiniMaxH3-PromptBuilder", "https://github.com/Adudeguyman/ComfyUI-Fantastic-MiniMaxH3-PromptBuilder.git"),\n    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),\n    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),\n    ("comfyui-websocket-reconnect-fix", "https://github.com/D3voz/comfyui-websocket-reconnect-fix.git"),\n    ("SolarRenderer", "https://github.com/Enderfga/SolarRenderer.git"),\n    ("comfyui_controlnet_aux", "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),\n    ("ComfyUI-H3-FaceRefine", "https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git"),\n    ("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),\n    ("comfyui-image-detail-contrast", "https://github.com/D3voz/comfyui-image-detail-contrast.git"),\n    ("Orion4D_FXMax", "https://github.com/orion4d/Orion4D_FXMax.git"),\n    ("ComfyUI-Spectrum-MiniMax-H3", "https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3.git"),\n)\n\nCORE_MODELS: tuple[dict[str, object], ...] = (\n    {\n        "directory": "diffusion_models",\n        "name": "minimax_h3_ref2va_pruned_int8_convrot.safetensors",\n        "approx_gb": 20.0,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",\n        ),\n    },\n    {\n        "directory": "text_encoders",\n        "name": "qwen3vl_32b_minimax_h3_int4_convrot.safetensors",\n        "approx_gb": 15.0,\n        "urls": (\n            "https://huggingface.co/Merserk/MiniMax-H3-INT4-ConvRot/resolve/main/qwen3vl_32b_minimax_h3_int4_convrot.safetensors",\n            "https://huggingface.co/Abiray/MiniMax-H3-GGUF/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int4_convrot.safetensors",\n        ),\n    },\n    {\n        "directory": "text_encoders",\n        "name": "qwen3vl_8b_int8_convrot.safetensors",\n        "approx_gb": 8.9,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",\n        ),\n    },\n    {\n        "directory": "vae",\n        "name": "minimax_h3_video_vae_int8_convrot.safetensors",\n        "approx_gb": 2.8,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors",\n        ),\n    },\n    {\n        "directory": "vae",\n        "name": "minimax_h3_audio_vae_fp32.safetensors",\n        "approx_gb": 0.6,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",\n        ),\n    },\n    {\n        "directory": "loras",\n        "name": "minimax_h3_DMAD_4step_full_lora_avg_rank_39_bf16.safetensors",\n        "approx_gb": 0.55,\n        "urls": (\n            "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/loras/minimax_h3_DMAD_4step_full_lora_avg_rank_39_bf16.safetensors",\n        ),\n    },\n    {\n        "directory": "checkpoints",\n        "name": "sam3.1_multiplex_fp16.safetensors",\n        "approx_gb": 1.7,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/sam3.1/resolve/main/checkpoints/sam3.1_multiplex_fp16.safetensors",\n        ),\n    },\n)\n\nEXTRA_FULL_MODELS: tuple[dict[str, object], ...] = (\n    {\n        "directory": "text_encoders",\n        "name": "qwen3.5_9b_flux3_prompt_upsampler_int8_convrot.safetensors",\n        "approx_gb": 9.8,\n        "urls": (\n            "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/text_encoders/qwen3.5_9b_flux3_prompt_upsampler_int8_convrot.safetensors",\n            "https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/text_encoders/qwen3.5_9b_flux3_prompt_upsampler_int8_convrot.safetensors",\n        ),\n    },\n    {\n        "directory": "diffusion_models",\n        "name": "flux3_image_klein_sd_int8_convrot.safetensors",\n        "approx_gb": 9.4,\n        "urls": (\n            "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-int8-convrot.safetensors",\n            "https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-int8-convrot.safetensors",\n        ),\n    },\n    {\n        "directory": "text_encoders",\n        "name": "qwen3vl_4b_fp8_scaled.safetensors",\n        "approx_gb": 5.24,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors",\n        ),\n    },\n    {\n        "directory": "loras",\n        "name": "solarrenderer_4step_lora.safetensors",\n        "approx_gb": 4.15,\n        "urls": (\n            "https://huggingface.co/Enderfga/SolarRenderer/resolve/main/solarrenderer_4step_lora.safetensors",\n        ),\n    },\n    {\n        "directory": "loras",\n        "name": "solarrenderer_control_lora.safetensors",\n        "approx_gb": 4.15,\n        "urls": (\n            "https://huggingface.co/Enderfga/SolarRenderer/resolve/main/solarrenderer_control_lora.safetensors",\n        ),\n    },\n    {\n        "directory": "sams",\n        "name": "sam_vit_b_01ec64.pth",\n        "approx_gb": 0.38,\n        "urls": (\n            "https://huggingface.co/comfyuistudio/sam_vit_b_01ec64.pth/resolve/main/sam_vit_b_01ec64.pth",\n            "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",\n        ),\n    },\n    {\n        "directory": "vae",\n        "name": "flux2-vae.safetensors",\n        "approx_gb": 0.34,\n        "urls": (\n            "https://huggingface.co/Comfy-Org/Flux-2-Klein-9B/resolve/main/split_files/vae/flux2-vae.safetensors",\n            "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/vae/flux2-vae.safetensors",\n        ),\n    },\n    {\n        "directory": "ultralytics/segm",\n        "name": "person_yolov8m-seg.pt",\n        "approx_gb": 0.06,\n        "urls": (\n            "https://huggingface.co/Bingsu/adetailer/resolve/main/person_yolov8m-seg.pt",\n        ),\n    },\n    {\n        "directory": "ultralytics/bbox",\n        "name": "face_yolov8m.pt",\n        "approx_gb": 0.05,\n        "urls": (\n            "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt",\n        ),\n    },\n)\n\nMODEL_PROFILES: dict[str, tuple[dict[str, object], ...]] = {\n    "core": CORE_MODELS,\n    "full": CORE_MODELS + EXTRA_FULL_MODELS,\n}\n\nEXTRA_NODE_PACKAGES: tuple[str, ...] = (\n    "huggingface_hub",\n    "hf_xet",\n    "av>=17.0.0",\n    "imageio-ffmpeg",\n    "color-matcher",\n    "mss",\n    "segment-anything",\n    "scikit-image",\n    "scikit-learn",\n    "piexif",\n    "dill",\n    "matplotlib",\n    "ultralytics",\n    "scipy",\n    "insightface",\n    "onnxruntime-gpu",\n    "scenedetect>=0.7",\n    "mediapipe>=0.8.0",\n    "omegaconf",\n    "addict",\n    "yacs",\n    "trimesh",\n    "albumentationsx",\n)\n\n\ndef olog(msg: str) -> None:\n    print(time.strftime("[%F %T] [openjutsu] ") + msg, flush=True)\n\n\ndef validate_model_destination(directory: str, name: str, comfy_dir: Path) -> Path:\n    """Resolve and verify that <comfy_dir>/models/<directory>/<name> stays strictly inside models/."""\n    models_root = (comfy_dir / "models").resolve()\n    rel_dir = (directory or "").strip().strip("/")\n    rel_name = (name or "").strip().lstrip("/")\n    if not rel_dir or not rel_name or "\\x00" in rel_dir or "\\x00" in rel_name:\n        raise ValueError(f"Invalid model destination: directory={directory!r}, name={name!r}")\n    if ".." in Path(rel_dir).parts or ".." in Path(rel_name).parts or "/" in rel_name or "\\\\" in rel_name:\n        raise ValueError(f"Refusing path traversal in model destination: {rel_dir}/{rel_name}")\n    dest = (models_root / rel_dir / rel_name).resolve()\n    if not dest.is_relative_to(models_root) or dest == models_root:\n        raise ValueError(f"Refusing model path outside {models_root}: {rel_dir}/{rel_name}")\n    return dest\n\n\ndef models_for_profile(profile: str) -> tuple[dict[str, object], ...]:\n    key = (profile or "full").strip().lower()\n    if key not in MODEL_PROFILES:\n        raise ValueError(f"Unsupported Openjutsu model profile {profile!r}; expected one of {sorted(MODEL_PROFILES)}")\n    return MODEL_PROFILES[key]\n\n\ndef render_extra_model_paths(comfy_dir: Path) -> str:\n    """Register `ultralytics` and `sams` model folders before custom nodes are imported."""\n    base = str(comfy_dir.resolve())\n    return (\n        "openjutsu:\\n"\n        f"  base_path: {json.dumps(base)}\\n"\n        "  ultralytics: models/ultralytics\\n"\n        "  ultralytics_bbox: models/ultralytics/bbox\\n"\n        "  ultralytics_segm: models/ultralytics/segm\\n"\n        "  sams: models/sams\\n"\n    )\n\n\nVRAM_MODES = ("auto", "highvram", "dynamic", "gpu-only")\nCHAINS_STORAGE_MODES = ("ssd", "drive")\n\n\ndef detect_gpu_vram_mib() -> int:\n    """Return total GPU0 VRAM in MiB, or 0 if unavailable."""\n    try:\n        out = subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],\n            text=True,\n            stderr=subprocess.DEVNULL,\n            timeout=15,\n        )\n        first = out.splitlines()[0].strip()\n        return int(float(first))\n    except Exception:\n        return 0\n\n\ndef select_comfy_vram_flags(vram_total_mib: int, mode: str = "auto") -> list[str]:\n    """Select ComfyUI VRAM flags; enables `--highvram` automatically on >=76 GiB GPUs (e.g. G4 96 GB)."""\n    clean = (mode or "auto").strip().lower()\n    if clean not in VRAM_MODES:\n        raise ValueError(f"Unsupported VRAM mode {mode!r}; expected one of {VRAM_MODES}")\n    if clean == "highvram":\n        return ["--highvram"]\n    if clean == "gpu-only":\n        return ["--gpu-only"]\n    if clean == "dynamic":\n        return []\n    return ["--highvram"] if vram_total_mib >= 76000 else []\n\n\ndef configure_chains_storage(work_dir: Path, data_dir: Path, mode: str = "ssd") -> str:\n    """Route `output/h3_chains` per-clip checkpoints (~450 MB/clip) to local NVMe SSD via VFS bind mount.\n\n    Using `mount --bind` (instead of a symlink) keeps `os.path.realpath` inside `output/`,\n    satisfying Context-Loop\'s `CheckpointGraphManager` path-confinement check while avoiding\n    multi-GB synchronous FUSE writes and Google Drive quota bloat.\n    """\n    clean = (mode or "ssd").strip().lower()\n    if clean not in CHAINS_STORAGE_MODES:\n        raise ValueError(f"Unsupported chains storage mode {mode!r}; expected one of {CHAINS_STORAGE_MODES}")\n    drive_chains = data_dir / "output" / "h3_chains"\n    drive_chains.mkdir(parents=True, exist_ok=True)\n    if clean == "drive":\n        if os.path.ismount(drive_chains):\n            subprocess.run(["umount", str(drive_chains)], check=False, timeout=30)\n            olog(f"Unmounted SSD bind mount; h3_chains stored on Drive: {drive_chains}")\n        return "drive"\n    if not str(data_dir).startswith("/content/drive/"):\n        return "ssd-native"\n    local_chains = work_dir / "h3_chains"\n    local_chains.mkdir(parents=True, exist_ok=True)\n    if os.path.ismount(drive_chains):\n        return "ssd-bind"\n    res = subprocess.run(["mount", "--bind", str(local_chains), str(drive_chains)],\n                         capture_output=True, text=True, check=False, timeout=30)\n    if res.returncode == 0:\n        olog(f"Bind-mounted local NVMe SSD for h3_chains checkpoints: {local_chains} -> {drive_chains}")\n        return "ssd-bind"\n    olog(f"Bind-mount unavailable ({res.stderr.strip() or res.returncode}); keeping h3_chains on {drive_chains}")\n    return "drive-fallback"\n\n\ndef normalize_linux_workflows(app_dir: Path) -> list[str]:\n    """Normalize Windows backslash relative model paths (e.g. face.api.json) for POSIX ComfyUI."""\n    wf_root = app_dir / "workflows"\n    updated: list[str] = []\n    if not wf_root.is_dir():\n        return updated\n    for wf_path in sorted(wf_root.rglob("*.json")):\n        raw = wf_path.read_text(encoding="utf-8")\n        data = json.loads(raw)\n        changed = False\n        if isinstance(data, dict):\n            for node in data.values():\n                if not isinstance(node, dict) or node.get("class_type") != "H3FaceTrackCrop":\n                    continue\n                inputs = node.get("inputs")\n                if not isinstance(inputs, dict):\n                    continue\n                for field in ("detector", "fallback_detector"):\n                    val = inputs.get(field)\n                    if isinstance(val, str) and "\\\\" in val:\n                        inputs[field] = val.replace("\\\\", "/")\n                        changed = True\n        if changed:\n            wf_path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n            updated.append(str(wf_path.relative_to(app_dir)))\n    return updated\n\n\ndef run_logged(cmd: list[str], log_path: Path, *, env: dict[str, str] | None = None,\n               cwd: Path | str | None = None, check: bool = True, timeout: int = 1800) -> subprocess.CompletedProcess[str]:\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    with log_path.open("a", encoding="utf-8") as handle:\n        handle.write(f"\\n$ {\' \'.join(str(x) for x in cmd)}\\n")\n        handle.flush()\n        res = subprocess.run(cmd, stdout=handle, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True, timeout=timeout)\n    if check and res.returncode != 0:\n        tail = "".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-45:])\n        raise RuntimeError(f"Command failed ({res.returncode}): {\' \'.join(str(x) for x in cmd[:5])}\\n{tail}")\n    return res\n\n\ndef ensure_nodejs(work_dir: Path, log_path: Path) -> Path:\n    node_dir = work_dir / f"node-v{NODE_VERSION}"\n    node_bin = node_dir / "bin" / "node"\n    if node_bin.is_file():\n        return node_dir / "bin"\n    arch = {"x86_64": "x64", "aarch64": "arm64"}.get(platform.machine())\n    if not arch:\n        raise RuntimeError(f"Unsupported architecture for Node.js binary: {platform.machine()}")\n    tar_name = f"node-v{NODE_VERSION}-linux-{arch}.tar.xz"\n    base_url = f"https://nodejs.org/dist/v{NODE_VERSION}/"\n    olog(f"Installing Node.js v{NODE_VERSION} ({arch}) -> {node_dir}")\n    with tempfile.TemporaryDirectory(dir=work_dir) as tmp:\n        archive = Path(tmp) / tar_name\n        req_sums = urllib.request.Request(base_url + "SHASUMS256.txt", headers={"User-Agent": "Mozilla/5.0"})\n        with urllib.request.urlopen(req_sums, timeout=30) as resp:\n            shasums = resp.read().decode("utf-8")\n        expected = next(\n            (line.split()[0] for line in shasums.splitlines() if line.split() and line.split()[-1] == tar_name),\n            None,\n        )\n        if not expected:\n            raise RuntimeError(f"Could not find SHA256 for {tar_name} in Node.js SHASUMS256.txt")\n        req_tar = urllib.request.Request(base_url + tar_name, headers={"User-Agent": "Mozilla/5.0"})\n        h = hashlib.sha256()\n        with urllib.request.urlopen(req_tar, timeout=120) as src, archive.open("wb") as dst:\n            while chunk := src.read(4 * 1024 * 1024):\n                dst.write(chunk)\n                h.update(chunk)\n        if h.hexdigest() != expected:\n            raise RuntimeError(f"Node.js archive SHA256 mismatch: expected {expected}, got {h.hexdigest()}")\n        extract_root = Path(tmp) / "extracted"\n        extract_root.mkdir()\n        with tarfile.open(archive, "r:xz") as tf:\n            tf.extractall(extract_root, filter="data")\n        extracted_sub = extract_root / f"node-v{NODE_VERSION}-linux-{arch}"\n        if node_dir.exists():\n            shutil.rmtree(node_dir)\n        extracted_sub.replace(node_dir)\n    ver = subprocess.check_output([str(node_bin), "--version"], text=True, timeout=15).strip()\n    olog(f"Node.js ready: {ver}")\n    return node_dir / "bin"\n\n\ndef _symlink_dir(local_path: Path, target_path: Path) -> None:\n    target_path.mkdir(parents=True, exist_ok=True)\n    local_path.parent.mkdir(parents=True, exist_ok=True)\n    if local_path.is_symlink():\n        if os.readlink(local_path) != str(target_path):\n            local_path.unlink()\n            local_path.symlink_to(target_path)\n    elif local_path.is_dir():\n        subprocess.run(["cp", "-rn", f"{local_path}/.", f"{target_path}/"], check=False, timeout=120)\n        shutil.rmtree(local_path)\n        local_path.symlink_to(target_path)\n    else:\n        local_path.symlink_to(target_path)\n\n\ndef prepare_comfy_tree(work_dir: Path, comfy_dir: Path, data_dir: Path) -> Path:\n    """Ensure ComfyUI repo, local SSD models/ root, and Drive symlinks exist before parallel tasks start."""\n    env_log = work_dir / "env-setup.log"\n    comfy_repo = os.environ.get("CT_COMFY_REPO", "https://github.com/comfyanonymous/ComfyUI.git")\n    if not (comfy_dir / ".git").is_dir():\n        if comfy_dir.exists():\n            shutil.rmtree(comfy_dir)\n        olog(f"Cloning ComfyUI -> {comfy_dir}")\n        run_logged(["git", "clone", "--depth", "1", comfy_repo, str(comfy_dir)], env_log, timeout=600)\n    else:\n        run_logged(["git", "-C", str(comfy_dir), "pull", "--ff-only"], env_log, check=False, timeout=300)\n\n    if (comfy_dir / "models").is_symlink():\n        (comfy_dir / "models").unlink()\n    (comfy_dir / "models" / "loras").mkdir(parents=True, exist_ok=True)\n    (comfy_dir / "custom_nodes").mkdir(parents=True, exist_ok=True)\n    for sub in ("output", "input", "user"):\n        _symlink_dir(comfy_dir / sub, data_dir / sub)\n    _symlink_dir(comfy_dir / "models" / "loras" / "drive", data_dir / "loras")\n    configure_chains_storage(work_dir, data_dir, os.environ.get("CT_OPENJUTSU_CHAINS_STORAGE", "ssd"))\n\n    extra_paths_file = work_dir / "extra_model_paths.yaml"\n    extra_paths_file.write_text(render_extra_model_paths(comfy_dir), encoding="utf-8")\n    return extra_paths_file\n\n\ndef _normalize_git_url(url: str) -> str:\n    return url.strip().rstrip("/").removesuffix(".git")\n\n\ndef sync_app_checkout(app_dir: Path, app_repo: str, env_log: Path) -> None:\n    repo = (app_repo or DEFAULT_OPENJUTSU_REPO).strip() or DEFAULT_OPENJUTSU_REPO\n    if (app_dir / ".git").is_dir():\n        origin_res = subprocess.run(\n            ["git", "-C", str(app_dir), "remote", "get-url", "origin"],\n            capture_output=True,\n            text=True,\n            timeout=15,\n        )\n        current_origin = origin_res.stdout.strip() if origin_res.returncode == 0 else ""\n        if current_origin and _normalize_git_url(current_origin) == _normalize_git_url(repo):\n            run_logged(["git", "-C", str(app_dir), "pull", "--ff-only"], env_log, check=False, timeout=300)\n            return\n        olog(f"Switching Openjutsu checkout ({current_origin or \'unknown\'}) -> {repo}")\n        shutil.rmtree(app_dir)\n    elif app_dir.exists():\n        shutil.rmtree(app_dir)\n    olog(f"Cloning Openjutsu ({repo}) -> {app_dir}")\n    run_logged(["git", "clone", "--depth", "1", repo, str(app_dir)], env_log, timeout=300)\n\n\ndef torch_supports_sm120(version_file: Path | None = None) -> bool:\n    """Check Blackwell sm_120 support via `torch/version.py` without cold-loading 2.5 GB of CUDA `.so`s\n    while `hf_xet` saturates NVMe disk writes; falls back to a 180s subprocess check if needed.\n    """\n    vf = version_file\n    if vf is None:\n        importlib.invalidate_caches()\n        spec = importlib.util.find_spec("torch")\n        if spec is None or not spec.origin:\n            return False\n        vf = Path(spec.origin).parent / "version.py"\n    if vf.is_file():\n        text = vf.read_text(encoding="utf-8", errors="replace")\n        if re.search(r"^\\s*cuda(?:\\s*:\\s*[^=]+)?\\s*=\\s*None\\b", text, flags=re.MULTILINE):\n            return False\n        m = re.search(r"""^\\s*cuda(?:\\s*:\\s*[^=]+)?\\s*=\\s*[\'"](\\d+)\\.(\\d+)""", text, flags=re.MULTILINE)\n        if m:\n            return (int(m.group(1)), int(m.group(2))) >= (12, 8)\n    try:\n        arch_check = subprocess.run(\n            [sys.executable, "-c", "import sys, torch; sys.exit(0 if \'sm_120\' in torch.cuda.get_arch_list() else 1)"],\n            capture_output=True,\n            text=True,\n            timeout=180,\n        )\n        return arch_check.returncode == 0\n    except subprocess.TimeoutExpired:\n        olog("[warn] Timed out probing torch.cuda.get_arch_list() after 180s; treating sm_120 as unsupported")\n        return False\n\n\ndef _modules_installed(*names: str) -> bool:\n    importlib.invalidate_caches()\n    return all(importlib.util.find_spec(name) is not None for name in names)\n\n\ndef setup_environment(work_dir: Path, comfy_dir: Path, data_dir: Path) -> dict[str, str]:\n    env_log = work_dir / "env-setup.log"\n    extra_paths_file = work_dir / "extra_model_paths.yaml"\n    if not extra_paths_file.is_file():\n        extra_paths_file = prepare_comfy_tree(work_dir, comfy_dir, data_dir)\n\n    if not shutil.which("ffmpeg"):\n        subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg >/dev/null 2>&1",\n                       shell=True, check=False, timeout=300)\n    if not shutil.which("uv"):\n        run_logged([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], env_log, timeout=300)\n    uv_bin = shutil.which("uv") or "uv"\n\n    # 1. Ensure Blackwell sm_120 PyTorch support if running on G4\n    try:\n        cap = subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True, timeout=15\n        ).splitlines()[0].strip()\n    except Exception:\n        cap = ""\n    if cap.startswith("12.") and not torch_supports_sm120():\n        olog("Upgrading PyTorch to CUDA 13.0 build for Blackwell sm_120...")\n        run_logged([\n            uv_bin, "pip", "install", "--system", "-U",\n            "torch", "torchvision", "torchaudio",\n            "--index-url", "https://download.pytorch.org/whl/cu130",\n        ], env_log, timeout=900)\n\n    # 3. Clone Openjutsu repo + Node.js 22 + build production Next.js app\n    app_dir = work_dir / "app"\n    app_repo = (os.environ.get("CT_OPENJUTSU_REPO") or DEFAULT_OPENJUTSU_REPO).strip()\n    sync_app_checkout(app_dir, app_repo, env_log)\n\n    patched_wfs = normalize_linux_workflows(app_dir)\n    if patched_wfs:\n        olog(f"Normalized POSIX detector paths in workflows: {\', \'.join(patched_wfs)}")\n\n    # 4. Clone all 13 Openjutsu custom node packs onto local SSD\n    olog(f"Installing {len(CUSTOM_NODES)} ComfyUI custom node packs...")\n    for node_name, node_repo in CUSTOM_NODES:\n        node_dir = comfy_dir / "custom_nodes" / node_name\n        if not (node_dir / ".git").is_dir():\n            if node_dir.exists():\n                shutil.rmtree(node_dir)\n            run_logged(["git", "clone", "--depth", "1", node_repo, str(node_dir)], env_log, timeout=300)\n        else:\n            run_logged(["git", "-C", str(node_dir), "pull", "--ff-only"], env_log, check=False, timeout=300)\n\n    # Apply Openjutsu\'s SolarRenderer extended-silence patch idempotently\n    solar_dir = comfy_dir / "custom_nodes" / "SolarRenderer"\n    solar_patch = app_dir / "patches" / "solarrenderer-long-clips.patch"\n    solar_nodes_py = solar_dir / "comfyui" / "nodes.py"\n    if solar_patch.is_file() and solar_nodes_py.is_file():\n        if "the held silence covers 170 frames" in solar_nodes_py.read_text(encoding="utf-8", errors="replace"):\n            chk = subprocess.run(["git", "-C", str(solar_dir), "apply", "--check", str(solar_patch)],\n                                 capture_output=True, text=True, timeout=30)\n            if chk.returncode == 0:\n                run_logged(["git", "-C", str(solar_dir), "apply", str(solar_patch)], env_log, timeout=30)\n                olog("Applied SolarRenderer extended-silence patch for long clips")\n\n    # 5. Install Python dependencies for ComfyUI + all custom nodes\n    deps_marker = work_dir / ".comfy_nodes_deps_v1.ok"\n    required_mods = ("comfy_kitchen", "insightface", "ultralytics", "segment_anything", "color_matcher")\n    if not deps_marker.is_file() or not _modules_installed(*required_mods):\n        olog("Installing ComfyUI and custom-node Python dependencies via uv...")\n        run_logged([uv_bin, "pip", "install", "--system", "-r", str(comfy_dir / "requirements.txt")],\n                   env_log, timeout=900)\n        run_logged([uv_bin, "pip", "install", "--system", *EXTRA_NODE_PACKAGES], env_log, timeout=900)\n        # sam2 for Impact-Pack (if not already importable)\n        if not _modules_installed("sam2"):\n            sam2_env = {**os.environ, "SAM2_BUILD_CUDA": "0"}\n            run_logged([uv_bin, "pip", "install", "--system", "sam2", "--no-build-isolation"],\n                       env_log, env=sam2_env, check=False, timeout=300)\n            if not _modules_installed("sam2"):\n                run_logged([uv_bin, "pip", "install", "--system", "--no-build-isolation",\n                            "git+https://github.com/facebookresearch/sam2"],\n                           env_log, env=sam2_env, check=False, timeout=600)\n        for node_name, _ in CUSTOM_NODES:\n            req = comfy_dir / "custom_nodes" / node_name / "requirements.txt"\n            if req.is_file() and node_name != "ComfyUI-Impact-Pack":\n                run_logged([uv_bin, "pip", "install", "--system", "-r", str(req)], env_log, check=False, timeout=600)\n        deps_marker.write_text("ok\\n", encoding="utf-8")\n\n    # 6. Dedicated virtualenv for comfy-api-proxy==0.1.6 + Openjutsu\'s live-progress patch_proxy.py\n    proxy_venv = work_dir / "venv-proxy"\n    proxy_py = proxy_venv / "bin" / "python"\n    proxy_bin = proxy_venv / "bin" / "comfy-api-proxy"\n    if not proxy_bin.is_file():\n        olog(f"Installing comfy-api-proxy=={PROXY_VERSION} -> {proxy_venv}")\n        run_logged([uv_bin, "venv", "--python", sys.executable, str(proxy_venv)], env_log, timeout=120)\n        run_logged([uv_bin, "pip", "install", "--python", str(proxy_py), f"comfy-api-proxy=={PROXY_VERSION}"],\n                   env_log, timeout=300)\n    run_logged([str(proxy_py), str(app_dir / "patch_proxy.py")], env_log, timeout=60)\n\n    # 7. Node.js 22 + Next.js production build\n    node_bin_dir = ensure_nodejs(work_dir, env_log)\n    node_env = {**os.environ, "PATH": f"{node_bin_dir}:{os.environ.get(\'PATH\', \'\')}", "NEXT_TELEMETRY_DISABLED": "1"}\n    (app_dir / ".env.local").write_text(\n        "COMFY_BASE_URL=http://127.0.0.1:8189\\n"\n        "COMFY_API_KEY=\\n"\n        "COMFYUI_URL=http://127.0.0.1:8188\\n"\n        "OPENJUTSU_PASSWORD=\\n",\n        encoding="utf-8",\n    )\n    head_sha = subprocess.check_output(["git", "-C", str(app_dir), "rev-parse", "HEAD"], text=True, timeout=15).strip()\n    build_marker = work_dir / ".next_build_sha"\n    if not (app_dir / ".next" / "BUILD_ID").is_file() or not build_marker.is_file() or build_marker.read_text().strip() != head_sha:\n        olog("Building Openjutsu Next.js production app (`npm ci && npm run build`)...")\n        run_logged([str(node_bin_dir / "npm"), "ci", "--no-audit", "--no-fund"], env_log,\n                   env=node_env, cwd=app_dir, timeout=600)\n        run_logged([str(node_bin_dir / "npm"), "run", "build"], env_log,\n                   env=node_env, cwd=app_dir, timeout=600)\n        build_marker.write_text(head_sha + "\\n", encoding="utf-8")\n\n    return {\n        "node_bin_dir": str(node_bin_dir),\n        "app_dir": str(app_dir),\n        "proxy_bin": str(proxy_bin),\n        "extra_model_paths": str(extra_paths_file),\n    }\n\n\ndef parse_hf_resolve_url(url: str) -> tuple[str, str, str] | None:\n    """Parse `https://huggingface.co/<repo_id>/(resolve|blob)/<revision>/<filename>`."""\n    m = HF_RESOLVE_URL_RE.match((url or "").strip())\n    if not m:\n        return None\n    return m.group(1), m.group(2), urllib.parse.unquote(m.group(3))\n\n\ndef _detect_system_ram_gib() -> float:\n    try:\n        for line in Path("/proc/meminfo").read_text(encoding="utf-8", errors="ignore").splitlines():\n            if line.startswith("MemTotal:"):\n                return int(line.split()[1]) / (1024.0 * 1024.0)\n    except OSError:\n        pass\n    return 0.0\n\n\ndef ensure_hf_downloader(work_dir: Path) -> str | None:\n    """Provision an isolated venv with `huggingface_hub` + `hf_xet` so parallel model downloads\n    saturate multi-Gbps links from t=0 without racing concurrent `--system` package installs.\n    """\n    if os.environ.get("CT_OPENJUTSU_DISABLE_XET") == "1":\n        return None\n    work_dir.mkdir(parents=True, exist_ok=True)\n    dl_venv = work_dir / "venv-hf-dl"\n    dl_py = dl_venv / "bin" / "python"\n    env_log = work_dir / "hf-dl-setup.log"\n    if dl_py.is_file():\n        chk = subprocess.run([str(dl_py), "-c", "import huggingface_hub, hf_xet"], capture_output=True, timeout=20)\n        if chk.returncode == 0:\n            return str(dl_py)\n    uv_bin = shutil.which("uv")\n    try:\n        if uv_bin:\n            if not dl_py.is_file():\n                run_logged([uv_bin, "venv", "--python", sys.executable, str(dl_venv)], env_log, timeout=120)\n            run_logged([uv_bin, "pip", "install", "--python", str(dl_py), "-U", "huggingface_hub", "hf_xet"],\n                       env_log, timeout=180)\n        else:\n            if not dl_py.is_file():\n                run_logged([sys.executable, "-m", "venv", str(dl_venv)], env_log, timeout=120)\n            run_logged([str(dl_py), "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"],\n                       env_log, timeout=300)\n        return str(dl_py)\n    except Exception as err:\n        olog(f"[model] Isolated hf_xet venv unavailable ({err}); checking system Python...")\n        chk = subprocess.run([sys.executable, "-c", "import huggingface_hub, hf_xet"], capture_output=True, timeout=20)\n        return sys.executable if chk.returncode == 0 else None\n\n\ndef _stage_dir_progress_bytes(stage_dir: Path) -> int:\n    best = 0\n    if not stage_dir.is_dir():\n        return 0\n    for p in stage_dir.rglob("*"):\n        try:\n            if p.is_file() and not p.is_symlink():\n                best = max(best, p.stat().st_size)\n        except OSError:\n            continue\n    return best\n\n\ndef _download_via_hf_xet(url: str, dest: Path, name: str, dl_python: str, hf_tok: str | None) -> int:\n    parsed = parse_hf_resolve_url(url)\n    if not parsed:\n        raise ValueError(f"Not a Hugging Face resolve URL: {url}")\n    repo_id, revision, filename = parsed\n    stage_dir = dest.parent / f".hf-stage-{dest.name}"\n    stage_dir.mkdir(parents=True, exist_ok=True)\n    dl_env = {\n        **os.environ,\n        "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0",\n        "HF_HUB_DISABLE_TELEMETRY": "1",\n        "HF_HUB_DISABLE_PROGRESS_BARS": "1",\n    }\n    high_perf = os.environ.get("CT_HF_XET_HIGH_PERFORMANCE")\n    if high_perf == "1" or (high_perf != "0" and _detect_system_ram_gib() >= 24.0):\n        dl_env["HF_XET_HIGH_PERFORMANCE"] = "1"\n    else:\n        dl_env.pop("HF_XET_HIGH_PERFORMANCE", None)\n    if hf_tok:\n        dl_env["HF_TOKEN"] = hf_tok\n        dl_env["HUGGING_FACE_HUB_TOKEN"] = hf_tok\n\n    code = (\n        "import os, sys\\n"\n        "from pathlib import Path\\n"\n        "from huggingface_hub import hf_hub_download\\n"\n        "repo_id, filename, revision, stage_dir, dest = sys.argv[1:6]\\n"\n        "tok = os.environ.get(\'HF_TOKEN\') or None\\n"\n        "p = Path(hf_hub_download(repo_id=repo_id, filename=filename, revision=revision, local_dir=stage_dir, token=tok)).resolve()\\n"\n        "os.replace(p, dest)\\n"\n    )\n    t0 = last_log = time.monotonic()\n    proc = subprocess.Popen(\n        [dl_python, "-c", code, repo_id, filename, revision, str(stage_dir), str(dest)],\n        stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT,\n        text=True,\n        env=dl_env,\n    )\n    try:\n        while proc.poll() is None:\n            now = time.monotonic()\n            if now - t0 > 1800:\n                proc.kill()\n                raise TimeoutError(f"hf_xet timed out after 1800s for {name}")\n            if now - last_log >= 15:\n                cur = _stage_dir_progress_bytes(stage_dir)\n                if cur > 0:\n                    mb_s = cur / 1e6 / max(now - t0, 0.1)\n                    olog(f"[model]   {name}: {cur/1e9:.2f} GB @ {mb_s:.0f} MB/s (hf_xet)")\n                last_log = now\n            time.sleep(1.0)\n        out = (proc.stdout.read() if proc.stdout else "").strip()\n        if proc.returncode != 0 or not (dest.is_file() and dest.stat().st_size > 0):\n            tail = out.splitlines()[-1] if out else f"exit code {proc.returncode}"\n            raise RuntimeError(f"hf_xet failed ({tail})")\n        shutil.rmtree(stage_dir, ignore_errors=True)\n        return dest.stat().st_size\n    finally:\n        if proc.poll() is None:\n            proc.kill()\n            proc.wait(timeout=5)\n\n\ndef _probe_http_range(url: str, headers: dict[str, str]) -> tuple[int, bool]:\n    probe_headers = {**headers, "Range": "bytes=0-0"}\n    req = urllib.request.Request(url, headers=probe_headers)\n    with urllib.request.urlopen(req, timeout=60) as resp:\n        if resp.status == 206:\n            cr = resp.headers.get("Content-Range") or ""\n            m = re.search(r"/(\\d+)\\s*$", cr)\n            if m:\n                return int(m.group(1)), True\n        total = int(resp.headers.get("Content-Length") or 0)\n        return total, False\n\n\ndef _download_via_http(\n    url: str,\n    dest: Path,\n    headers: dict[str, str],\n    name: str,\n    *,\n    parallel_threshold: int = 32 * 1024 * 1024,\n    range_workers: int = 8,\n) -> int:\n    partial = dest.with_name(f".{dest.name}.{os.getpid()}.partial")\n    t0 = time.monotonic()\n\n    # Fast path 1: Multi-connection parallel HTTP Range (`os.pwrite`) for large files\n    if not partial.exists():\n        try:\n            total, supports_range = _probe_http_range(url, headers)\n        except Exception:\n            total, supports_range = 0, False\n        if supports_range and total >= parallel_threshold and range_workers > 1 and hasattr(os, "pwrite"):\n            fd = os.open(partial, os.O_CREAT | os.O_RDWR | os.O_TRUNC, 0o644)\n            try:\n                os.ftruncate(fd, total)\n                part_size = (total + range_workers - 1) // range_workers\n                slices = [(i * part_size, min(total - 1, (i + 1) * part_size - 1))\n                          for i in range(range_workers) if i * part_size < total]\n                lock = threading.Lock()\n                progress = [0, t0]  # [copied_bytes, last_log_monotonic]\n\n                def _fetch_slice(bounds: tuple[int, int]) -> None:\n                    start, end = bounds\n                    cur = start\n                    for sub_try in range(1, 4):\n                        try:\n                            req = urllib.request.Request(url, headers={**headers, "Range": f"bytes={cur}-{end}"})\n                            with urllib.request.urlopen(req, timeout=120) as resp:\n                                if resp.status != 206:\n                                    raise IOError(f"Expected HTTP 206 for Range request, got {resp.status}")\n                                while cur <= end:\n                                    chunk = resp.read(min(4 * 1024 * 1024, end - cur + 1))\n                                    if not chunk:\n                                        break\n                                    os.pwrite(fd, chunk, cur)\n                                    cur += len(chunk)\n                                    with lock:\n                                        progress[0] += len(chunk)\n                                        now = time.monotonic()\n                                        if now - progress[1] >= 15:\n                                            mb_s = progress[0] / 1e6 / max(now - t0, 0.1)\n                                            olog(f"[model]   {name}: {progress[0]/1e9:.2f}/{total/1e9:.2f} GB "\n                                                 f"({progress[0]*100/total:.0f}%) @ {mb_s:.0f} MB/s (http-x{len(slices)})")\n                                            progress[1] = now\n                            if cur != end + 1:\n                                raise IOError(f"Incomplete Range slice {cur}/{end + 1}")\n                            return\n                        except Exception:\n                            if sub_try == 3:\n                                raise\n                            time.sleep(sub_try * 2)\n\n                with ThreadPoolExecutor(max_workers=len(slices)) as pool:\n                    futs = [pool.submit(_fetch_slice, b) for b in slices]\n                    for fut in as_completed(futs):\n                        fut.result()\n            except Exception:\n                os.close(fd)\n                partial.unlink(missing_ok=True)\n                raise\n            else:\n                os.close(fd)\n                partial.replace(dest)\n                return total\n\n    # Fast path 2: Resumable single-stream HTTP download (never discards partial progress on reset)\n    existing = partial.stat().st_size if partial.is_file() else 0\n    req_headers = {**headers, "Range": f"bytes={existing}-"} if existing > 0 else dict(headers)\n    req = urllib.request.Request(url, headers=req_headers)\n    last_log = t0\n    with urllib.request.urlopen(req, timeout=120) as resp:\n        resumed = existing > 0 and resp.status == 206\n        mode = "ab" if resumed else "wb"\n        base_bytes = existing if resumed else 0\n        rem = int(resp.headers.get("Content-Length") or 0)\n        total = (base_bytes + rem) if rem > 0 else 0\n        copied = base_bytes\n        with partial.open(mode) as out:\n            while True:\n                chunk = resp.read(16 * 1024 * 1024)\n                if not chunk:\n                    break\n                out.write(chunk)\n                copied += len(chunk)\n                now = time.monotonic()\n                if now - last_log >= 15:\n                    mb_s = (copied - base_bytes) / 1e6 / max(now - t0, 0.1)\n                    pct = f" ({copied*100/total:.0f}%)" if total > 0 else ""\n                    olog(f"[model]   {name}: {copied/1e9:.2f}/{total/1e9:.2f} GB{pct} @ {mb_s:.0f} MB/s")\n                    last_log = now\n    if total > 0 and copied != total:\n        raise IOError(f"Incomplete read for {name}: got {copied} of {total} bytes")\n    if copied == 0:\n        raise IOError(f"Empty download for {name}")\n    partial.replace(dest)\n    return copied\n\n\ndef _download_single_model(item: dict[str, object], comfy_dir: Path, dl_python: str | None = None) -> dict[str, object]:\n    directory = str(item["directory"])\n    name = str(item["name"])\n    urls = tuple(str(u) for u in item["urls"])  # type: ignore[arg-type]\n    dest = validate_model_destination(directory, name, comfy_dir)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    if dest.is_file() and dest.stat().st_size > 0:\n        sz = dest.stat().st_size\n        olog(f"[model] Cached ({sz/1e9:.2f} GB): {directory}/{name}")\n        return {"model": f"{directory}/{name}", "status": "cached", "size": sz}\n\n    hf_tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")\n    last_err: Exception | None = None\n    partial = dest.with_name(f".{dest.name}.{os.getpid()}.partial")\n    stage_dir = dest.parent / f".hf-stage-{dest.name}"\n\n    for url in urls:\n        partial.unlink(missing_ok=True)\n        shutil.rmtree(stage_dir, ignore_errors=True)\n        headers = {"User-Agent": "Mozilla/5.0"}\n        if hf_tok and "huggingface.co" in url:\n            headers["Authorization"] = f"Bearer {hf_tok}"\n        use_xet = bool(dl_python and parse_hf_resolve_url(url))\n        for attempt in range(1, 4):\n            t0 = time.monotonic()\n            try:\n                if use_xet and attempt <= 2:\n                    assert dl_python is not None\n                    copied = _download_via_hf_xet(url, dest, name, dl_python, hf_tok)\n                    engine = "hf_xet"\n                else:\n                    copied = _download_via_http(url, dest, headers, name)\n                    engine = "http"\n                dt = max(time.monotonic() - t0, 0.1)\n                olog(f"[model] Downloaded ({copied/1e9:.2f} GB in {dt:.1f}s, {copied/1e6/dt:.0f} MB/s, {engine}): {directory}/{name}")\n                return {"model": f"{directory}/{name}", "status": "downloaded", "size": copied}\n            except Exception as err:\n                last_err = err\n                olog(f"[model] Attempt {attempt}/3 failed for {name} ({url}): {err}")\n                time.sleep(min(3 * attempt, 10))\n        partial.unlink(missing_ok=True)\n        shutil.rmtree(stage_dir, ignore_errors=True)\n    raise RuntimeError(f"Failed to download {directory}/{name}: {last_err}")\n\n\ndef stage_models(\n    comfy_dir: Path,\n    profile: str = "full",\n    workers: int = 3,\n    work_dir: Path | None = None,\n) -> list[dict[str, object]]:\n    # Schedule largest multi-GB weights first (LPT) so tail stragglers never hold up readiness.\n    items = sorted(models_for_profile(profile), key=lambda m: float(m.get("approx_gb", 0.0)), reverse=True)  # type: ignore[arg-type]\n    free_gb = shutil.disk_usage(comfy_dir.parent if not comfy_dir.exists() else comfy_dir).free / 1e9\n    olog(f"Staging {len(items)} model files (profile={profile}, free SSD={free_gb:.1f} GB)...")\n\n    needs_hf = any(\n        not (validate_model_destination(str(m["directory"]), str(m["name"]), comfy_dir).is_file())\n        and any(parse_hf_resolve_url(str(u)) for u in m["urls"])  # type: ignore[attr-defined]\n        for m in items\n    )\n    dl_python = ensure_hf_downloader(work_dir or (comfy_dir.parent / "openjutsu")) if needs_hf else None\n\n    results: list[dict[str, object]] = []\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        future_map = {pool.submit(_download_single_model, item, comfy_dir, dl_python): item for item in items}\n        for fut in as_completed(future_map):\n            results.append(fut.result())\n    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):\n        shutil.rmtree(p, ignore_errors=True)\n    total_gb = sum(int(r["size"]) for r in results) / 1e9\n    olog(f"All {len(results)} model files ready on local SSD ({total_gb:.2f} GB total)")\n    return results\n\n\nDEFAULT_MODEL_SETTINGS: dict[str, object] = {\n    "model": "minimax_h3_ref2va_pruned_int8_convrot.safetensors",\n    "loras": [{"name": "minimax_h3_DMAD_4step_full_lora_avg_rank_39_bf16.safetensors", "strength": 1}],\n    "steps": 6,\n    "textEncoder": "qwen3vl_32b_minimax_h3_int4_convrot.safetensors",\n    "videoVae": "minimax_h3_video_vae_int8_convrot.safetensors",\n    "audioVae": "minimax_h3_audio_vae_fp32.safetensors",\n    "acceleration": "none",\n}\n\n\ndef setup_check_payloads(profile: str = "full") -> dict[str, dict[str, object]]:\n    """Return representative `/api/setup` payloads covering Openjutsu\'s workflows."""\n    payloads: dict[str, dict[str, object]] = {\n        "mask": {"kind": "mask", "hatFix": True, "margin": "normal", "target": "person"},\n        "people": {"kind": "people", "resolution": "736p"},\n        "fit": {"kind": "fit", "resolution": "736p"},\n        "color": {\n            "kind": "color",\n            "settings": {\n                "detail": -0.2,\n                "tone": {\n                    "mode": "Reinhard",\n                    "exposure": 0.09,\n                    "contrast": 0.84,\n                    "pivot": 0.69,\n                    "highlight_rolloff": 0.27,\n                    "shadow_lift": 0.15,\n                    "black_floor": 0.03,\n                    "saturation": 1.05,\n                    "strength": 0.2,\n                },\n            },\n        },\n        "simple": {\n            "kind": "simple",\n            "settings": {\n                "role": "",\n                "resolution": "736p",\n                "motion": 768,\n                "hatFix": True,\n                "noLipSync": True,\n                "margin": "normal",\n                "target": "person",\n                "seed": 0,\n                "clipSeconds": 6,\n            },\n            "model": DEFAULT_MODEL_SETTINGS,\n        },\n        "advanced": {\n            "kind": "advanced",\n            "pictures": 2,\n            "settings": {\n                "roles": ["", ""],\n                "resolution": "736p",\n                "motion": 768,\n                "hatFix": True,\n                "noLipSync": True,\n                "margin": "normal",\n                "seed": 0,\n                "clipSeconds": 6,\n            },\n            "model": DEFAULT_MODEL_SETTINGS,\n        },\n        "object": {\n            "kind": "object",\n            "picture": True,\n            "settings": {\n                "mode": "change",\n                "target": "jacket",\n                "changeTo": "leather jacket",\n                "biggestOnly": True,\n                "keepOutline": False,\n                "resolution": "736p",\n                "motion": 768,\n                "margin": "normal",\n                "seed": 0,\n                "clipSeconds": 6,\n            },\n            "model": DEFAULT_MODEL_SETTINGS,\n        },\n        "montage": {\n            "kind": "montage",\n            "pictures": 2,\n            "review": None,\n            "settings": {\n                "roles": ["", ""],\n                "resolution": "736p",\n                "motion": 768,\n                "hatFix": True,\n                "noLipSync": True,\n                "margin": "normal",\n                "seed": 0,\n                "clipSeconds": 6,\n            },\n            "model": DEFAULT_MODEL_SETTINGS,\n        },\n    }\n    if (profile or "full").strip().lower() == "full":\n        payloads["face"] = {\n            "kind": "face",\n            "settings": {"style": "mouth", "seed": 0, "chunkSeconds": 6.5},\n            "model": DEFAULT_MODEL_SETTINGS,\n        }\n    return payloads\n', encoding='utf-8')
if str(OJ_WORK) not in sys.path:
    sys.path.insert(0, str(OJ_WORK))
from colab_cache import restore_cache, save_cache
from openjutsu_runtime import (
    detect_gpu_vram_mib,
    olog,
    prepare_comfy_tree,
    select_comfy_vram_flags,
    setup_check_payloads,
    setup_environment,
    stage_models,
)

OJ_GPU_SIG = ct_gpu_stat('name,compute_cap')
OJ_CACHE_KEY = hashlib.sha256(f'{sys.version_info[:2]}:{platform.machine()}:{OJ_GPU_SIG}'.encode()).hexdigest()[:16]
OJ_CACHE_DIRS = [OJ_WORK / 'triton-cache', OJ_WORK / 'inductor-cache']
for _cache_dir in OJ_CACHE_DIRS:
    restore_cache(_cache_dir, Path(DATA_DIR) / 'legacy-caches' / _cache_dir.name,
                  Path(DATA_DIR) / 'kernel-archives', OJ_CACHE_KEY, log=olog)

prepare_comfy_tree(OJ_WORK, OJ_COMFY, Path(DATA_DIR))
olog(f'Starting parallel environment build and {OJ_PROFILE} model staging on {OJ_GPU_SIG}...')
with ThreadPoolExecutor(max_workers=2) as _pool:
    _f_env = _pool.submit(setup_environment, OJ_WORK, OJ_COMFY, Path(DATA_DIR))
    _f_models = _pool.submit(stage_models, OJ_COMFY, OJ_PROFILE, 3, OJ_WORK)
    while not (_f_env.done() and _f_models.done()):
        for _fut in (_f_env, _f_models):
            if _fut.done() and _fut.exception():
                raise _fut.exception()
        olog(f"Setup heartbeat | env={'done' if _f_env.done() else 'running'} "
             f"models={'done' if _f_models.done() else 'running'} | gpu={ct_gpu_stat()}")
        time.sleep(30)
    OJ_META = _f_env.result()
    OJ_MODELS = _f_models.result()
olog(f"Setup complete: {len(OJ_MODELS)} models staged, app={OJ_META['app_dir']}")


In [ ]:
# @title 2. Service -- ComfyUI (:8188) + patched comfy-api-proxy (:8189) + Openjutsu Next.js (:3000)
if "OJ_META" not in globals():
    raise RuntimeError("Openjutsu setup cell failed before completing environment build; see setup traceback above.")
OJ_PROXY_STATE = OJ_WORK / "proxy-state"
OJ_PROXY_STATE.mkdir(parents=True, exist_ok=True)

def _wait_proc_http(proc_key, url, timeout=600, ok=lambda c: c < 500):
    deadline = time.time() + timeout
    while time.time() < deadline:
        proc = PROCS[proc_key][0]
        if proc.poll() is not None:
            raise RuntimeError(f"{proc_key} exited early (code {proc.returncode}):\n{ct_tail(proc_key, 60)}")
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                if ok(resp.status):
                    return resp.status
        except urllib.error.HTTPError as err:
            if ok(err.code):
                return err.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{proc_key} ({url}) not ready after {timeout}s:\n{ct_tail(proc_key, 50)}")

def start_comfyui():
    subprocess.run(["pkill", "-f", f"{OJ_COMFY}/main.py"], check=False)
    vram_flags = select_comfy_vram_flags(
        detect_gpu_vram_mib(),
        os.environ.get("CT_COMFY_VRAM_MODE", "auto"),
    )
    olog(f"Starting ComfyUI (VRAM flags={vram_flags or ['dynamic-default']})")
    return ct_spawn(
        "comfyui",
        [
            sys.executable,
            str(OJ_COMFY / "main.py"),
            "--listen", "127.0.0.1",
            "--port", "8188",
            "--extra-model-paths-config", OJ_META["extra_model_paths"],
            "--cache-classic",
            *vram_flags,
        ],
        cwd=str(OJ_COMFY),
        env={
            "TRITON_CACHE_DIR": str(OJ_WORK / "triton-cache"),
            "TORCHINDUCTOR_CACHE_DIR": str(OJ_WORK / "inductor-cache"),
            "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        },
    )

def start_comfy_proxy():
    subprocess.run(["pkill", "-f", "comfy-api-proxy"], check=False)
    return ct_spawn(
        "comfy_proxy",
        [
            OJ_META["proxy_bin"],
            "--comfyui", "http://127.0.0.1:8188",
            "--host", "127.0.0.1",
            "--port", "8189",
            "--max-upload-mb", "500",
            "--state-dir", str(OJ_PROXY_STATE),
        ],
        cwd=str(OJ_WORK),
    )

def start_service():
    subprocess.run(["pkill", "-f", "next/dist/bin/next"], check=False)
    return ct_spawn(
        "service",
        [
            str(Path(OJ_META["app_dir"]) / "node_modules" / ".bin" / "next"),
            "start",
            "--hostname", "127.0.0.1",
            "--port", str(SERVICE_PORT),
        ],
        cwd=OJ_META["app_dir"],
        env={
            "PATH": f"{OJ_META['node_bin_dir']}:{os.environ.get('PATH', '')}",
            "NODE_ENV": "production",
            "NEXT_TELEMETRY_DISABLED": "1",
            "COMFY_BASE_URL": "http://127.0.0.1:8189",
            "COMFY_API_KEY": "",
            "COMFYUI_URL": "http://127.0.0.1:8188",
            "OPENJUTSU_PASSWORD": "",
        },
    )

PROCS["comfyui"] = (start_comfyui(), start_comfyui)
_wait_proc_http("comfyui", "http://127.0.0.1:8188/system_stats", timeout=600, ok=lambda c: c == 200)
_wait_proc_http("comfyui", "http://127.0.0.1:8188/websocket_reconnect_fix", timeout=60, ok=lambda c: c == 200)
olog("ComfyUI + custom nodes listening on 127.0.0.1:8188")

PROCS["comfy_proxy"] = (start_comfy_proxy(), start_comfy_proxy)
_wait_proc_http("comfy_proxy", "http://127.0.0.1:8189/", timeout=120, ok=lambda c: c < 500)
olog("Patched comfy-api-proxy listening on 127.0.0.1:8189")

PROCS["service"] = (start_service(), start_service)
_wait_proc_http("service", f"http://127.0.0.1:{SERVICE_PORT}/api/models", timeout=180, ok=lambda c: c == 200)
olog(f"Openjutsu Next.js server listening on 127.0.0.1:{SERVICE_PORT}")


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            header_up -X-Forwarded-Host
            header_up -X-Forwarded-Proto
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- verify /api/models, /api/setup & public Caddy auth before CT_READY
import base64

if "OJ_META" not in globals() or "service" not in PROCS:
    raise RuntimeError("Openjutsu service did not start because an earlier notebook cell failed; see traceback above.")

LOCAL_ORIGIN = f"http://127.0.0.1:{SERVICE_PORT}"

def _oj_local_json(path, payload=None, timeout=30):
    headers = {"Origin": LOCAL_ORIGIN, "Accept": "application/json"}
    data = None
    method = "GET"
    if payload is not None:
        method = "POST"
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(LOCAL_ORIGIN + path, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))

_, _models_info = _oj_local_json("/api/models", timeout=30)
if _models_info.get("error") or not _models_info.get("models"):
    raise RuntimeError(f"Openjutsu /api/models failed: {_models_info}\n{ct_tail('comfyui', 50)}")

for _mode, _payload in setup_check_payloads(OJ_PROFILE).items():
    _, _setup_res = _oj_local_json("/api/setup", payload=_payload, timeout=45)
    if not _setup_res.get("ok"):
        raise RuntimeError(
            f"Openjutsu /api/setup check failed for mode={_mode!r}: {_setup_res}\n{ct_tail('comfyui', 60)}"
        )
olog(f"Verified Openjutsu /api/models and /api/setup for profile={OJ_PROFILE}")

for _cache_dir in OJ_CACHE_DIRS:
    try:
        save_cache(_cache_dir, Path(DATA_DIR) / "kernel-archives", OJ_CACHE_KEY, log=olog)
    except (OSError, ValueError) as _err:
        olog(f"Cache archive save warning ({_cache_dir.name}): {_err}")

def _public_bearer_probe():
    req = urllib.request.Request(
        PUBLIC_URL + "/api/models",
        headers={"Authorization": f"Bearer {API_TOKEN}", "User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        body = json.loads(resp.read().decode("utf-8"))
        if body.get("error"):
            raise RuntimeError(f"Public /api/models returned error: {body}")
        return resp.status

def _verify_public_auth():
    # 1. Unauthenticated request must return 401 and must NOT leak Set-Cookie
    req_unauth = urllib.request.Request(PUBLIC_URL + "/", headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req_unauth, timeout=15) as resp:
            raise RuntimeError(f"Unauthenticated public request unexpectedly returned HTTP {resp.status}")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 without credentials, got HTTP {err.code}")
        if err.headers.get("Set-Cookie"):
            raise RuntimeError("Public 401 response leaked Set-Cookie")
    # 2. Invalid Bearer token must return 401
    req_bad = urllib.request.Request(
        PUBLIC_URL + "/api/models",
        headers={"Authorization": "Bearer invalid-probe-token", "User-Agent": "Mozilla/5.0"},
    )
    try:
        with urllib.request.urlopen(req_bad, timeout=15) as resp:
            raise RuntimeError(f"Invalid Bearer token unexpectedly returned HTTP {resp.status}")
    except urllib.error.HTTPError as err:
        if err.code != 401:
            raise RuntimeError(f"Expected HTTP 401 for invalid Bearer token, got HTTP {err.code}")
    # 3. Browser Basic Auth must return 200 and issue ct_session cookie
    basic = base64.b64encode(f"{AUTH_USER}:{AUTH_PASS}".encode("utf-8")).decode("ascii")
    req_basic = urllib.request.Request(
        PUBLIC_URL + "/",
        headers={"Authorization": f"Basic {basic}", "User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req_basic, timeout=20) as resp:
        if resp.status != 200:
            raise RuntimeError(f"Basic Auth request returned HTTP {resp.status}")

ct_public_gate(PUBLIC_URL, _public_bearer_probe, _verify_public_auth, olog, timeout=180)
if AUTH == "caddy" and not CREDS_FROM_ENV:
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def _oj_health():
    if PROCS["comfyui"][0].poll() is not None or PROCS["comfy_proxy"][0].poll() is not None:
        return False
    req = urllib.request.Request(LOCAL_ORIGIN + "/", headers={"Host": f"127.0.0.1:{SERVICE_PORT}"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        return resp.status == 200

def _oj_heartbeat(uptime_s):
    if uptime_s % 600 < 60:
        states = " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items())
        olog(f"[heartbeat] uptime={uptime_s}s {states} gpu={ct_gpu_stat()}")

ct_watchdog(_oj_health, start_service, olog, interval=60, grace=900, heartbeat=_oj_heartbeat)
